# JeDepth Training on Kaggle RTX Pro 6000

Setup môi trường, copy code, và huấn luyện mô hình JeDepth stereo depth estimation.

**Workflow:**
1. Import utility packages (pre-installed, no internet needed)
2. Copy code từ dataset → working dir
3. Setup data paths (symlink)
4. Chạy training với tensorboard, eval + test inference mỗi 5 epoch

In [ ]:
import sys, os

# Thêm utility packages vào path (cài sẵn từ jedepth-utility-script kernel)
UTILITY_PATH = "/kaggle/input/notebooks/laithetrung/hitnet-utility-script"
if os.path.exists(UTILITY_PATH):
    sys.path.append(UTILITY_PATH)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import multiprocessing

# Using os module
logical_cores_os = os.cpu_count()

# Using multiprocessing module (identical to os.cpu_count())
logical_cores_mp = multiprocessing.cpu_count()

if logical_cores_os is not None:
    print(f"Number of logical cores (os): {logical_cores_os}")
    print(f"Number of logical cores (multiprocessing): {logical_cores_mp}")
else:
    print("Could not determine the number of CPU cores.")


In [ ]:
# Copy JeDepth code từ dataset (read-only) sang working dir
import shutil

CODE_SRC = "/kaggle/input/datasets/laithetrung/jedepth-code/JeDepth"
WORK_DIR = "/kaggle/working/JeDepth"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(CODE_SRC, WORK_DIR)

os.chdir(WORK_DIR)
print(f"Working dir: {os.getcwd()}")
!ls

In [ ]:
# Setup data paths: symlink dataset vào data/processed_data
DATASET_SRC = "/kaggle/input/datasets/laithetrung/stereo-smallbaseline"

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/processed_data"):
    os.symlink(DATASET_SRC, "data/processed_data")
print(f"Symlinked: data/processed_data -> {DATASET_SRC}")

# Verify dataset
import pandas as pd
train_df = pd.read_csv("data/processed_data/train.csv")
val_df = pd.read_csv("data/processed_data/val.csv")
print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Example path:  {train_df.iloc[0]['left_image_path']}")

!ls data/processed_data/ | head -10

In [ ]:
# Verify imports trước khi training
sys.path.insert(0, WORK_DIR)

from jedepth.model.iinet import JeDepth
from jedepth.dataset.depth_dataset import StereoDepthDataset
from jedepth.evaluation.evaluation import evaluate as compute_metrics
from easydict import EasyDict

print("All JeDepth imports OK!")

In [ ]:
# ============================================================================
# Tạo config YAML trực tiếp trong notebook — sửa ở đây thay vì push code lên GitHub
# ============================================================================
import yaml

config = {
    "DATA": {
        "TRAIN_CSV": "data/processed_data/train.csv",
        "VAL_CSV": "data/processed_data/val.csv",
        "ROOT": "data/processed_data",
        "CROP_SIZE": [384, 512],
        "AUGMENT": False,
        "TEST_IMAGES": "test_images",
    },
    "MODEL": {
        "NAME": "JeDepth",
        "MAX_DISP": 192,
        "MATCHING_FEATURE_DIMS": 16,
        "CV_ENCODER_TYPE": "multi_scale_encoder",
        "MATCHING_SCALE": 2,
        "MULTISCALE": 2,
        "DEPTH_DECODER_NAME": "unet_pp",
        "OUT_SCALE": 4,
        "FEATURE_VOLUME_TYPE": "ms_cost_volume",
        "DISP_SCALE": 32,
        "DOT_DIM": 1,
        "MATCHING_ENCODER_TYPE": "unet",
        "REGRESSION_TOPK": 2,
        "FIND_UNUSED_PARAMETERS": True,
        "CKPT": -1,
        "PRETRAINED_MODEL": "",
        "BACKBONE_PRETRAINED": "pretrained/mobilenetv3_large_100.safetensors",
    },
    "OPTIMIZATION": {
        "BATCH_SIZE": 32,
        "AMP": True,
        "NUM_EPOCHS": 60,
        "OPTIMIZER": {
            "NAME": "AdamW",
            "LR": 0.0003,
            "WEIGHT_DECAY": 0.0001,
        },
        "SCHEDULER": {
            "NAME": "MultiStepLR",
            "GAMMA": 0.5,
            "MILESTONES": [20, 35, 45],
            "ON_EPOCH": True,
        },
    },
    "TRAINER": {
        "EVAL_INTERVAL": 1,
        "CKPT_SAVE_INTERVAL": 1,
        "MAX_CKPT_SAVE_NUM": 10,
        "LOG_INTERVAL": 10,
        "TRAIN_VISUALIZATION": False,
        "EVAL_VISUALIZATION": True,
        "UNCER_ONLY": True,
        "LOSS_WEIGHT": {
            "l1": [5, 2.5, 1.25, 0.6],
            "grad": [1.0, 1.0, 0.5, 0.5],
            "normal": 2.5,
            "focal": 1.0
        },
    },
}

# Ghi file config
cfg_path = "cfgs/iinet/kaggle_config.yaml"
os.makedirs(os.path.dirname(cfg_path), exist_ok=True)
with open(cfg_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config saved to: {cfg_path}")
print("---")


In [ ]:
# Chạy training với config vừa tạo ở cell trên
print("=== Starting Training ===")
!python train.py \
    --cfg cfgs/iinet/kaggle_config.yaml \
    --output_dir /kaggle/working/output \
    --workers 32 \
    --gpu 0 \
    --test_images test_images \
    --seed 142

In [ ]:
# Copy outputs sang /kaggle/working để download
import shutil

OUTPUT_SRC = "/kaggle/working/output"
LOGS_DST = "/kaggle/working/results"

if os.path.exists(OUTPUT_SRC):
    shutil.copytree(OUTPUT_SRC, LOGS_DST, dirs_exist_ok=True)
    print(f"Results copied to {LOGS_DST}")

    # List checkpoints
    import glob
    ckpts = sorted(glob.glob(f"{LOGS_DST}/**/ckpt/*.pth", recursive=True))
    print(f"\nCheckpoints ({len(ckpts)}):")
    for c in ckpts:
        size_mb = os.path.getsize(c) / 1024**2
        print(f"  {os.path.basename(c)} ({size_mb:.1f} MB)")
else:
    print("No output found - training may not have completed.")